# 01 - Understanding the ClimX Dataset structure and preprocessing

## Objective
The purpose of this notebook is to understand the structure and organizationof the ClimX dataset before developing a ML model.

## Questions 
1. What datasets are available?
2. Forcing and target variables?
3. Shape, dimensions, coordinates,and units?
4. How are time, latitude, longitude, and scenarios represented?
5. How are forcing variables aligned with target variables?

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [3]:
# Data directory
DATADIR = Path('../data/lite/')

for path in sorted(DATADIR.glob('*.nc')):
    print(path.name)

NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_forcings_all.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_hist_train_1850_1931.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_hist_train_1932_2014.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_test_forcings_ssp245.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_train_ssp126.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_train_ssp370.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_train_ssp585.nc


## Dataset Description

The lite data contains seven NetCDF files.

- Historical training simulations (1850-2014)
- Future training scenarios (SSP1-2.6, SSP3-7.0, SSP5-8.5)
- A held-out test scenario (SSP2-4.5)
- A combied forcing dataset

In Xarray, a Dataset is organized into four important components: 
1. Dimensions: How the data is structured.
2. Coordinates: Where or when the data is located
3. Data variables: The actual data values
4. Attributes: Metadata about the dataset

### Xarray Dataset

A python library for working with multi-dimensional, labeled scientific data, particularly useful for climate science, atmospheric physics, oceanography, and other fields involving data that varies across space and time.

Xarray objects:
- DataArray: Represent a single multidimensional variable, (e.g. temperature, precipitation, etc.)
- Dataset: A collection of DataArrays that can share dimensions and coordinates, such as temperature, precipitation, pressure, and humidity.

Key components:
1. Dimensions: named axes that organize the data (e.g. time, lat, lon)
2. Coordinates: Values or labels associated with with positions
3. Data variables: Actual scientific quantities stored (temperature, precessure etc)
4. Atributes: Metadata describing the data

# ClimX Dataset: exploring the variables

The ClimX dataset contains **17 data variables**, divided into two categories:

- **10 input variables (forcings):** External factors that influence Earth's climate.
- **7 target variables (outputs):** Climate quantities predicted by the NorESM2-MM model.

## 1. Greenhouse Gas Forcings

These variables represent atmospheric greenhouse gases that influence Earth's radiative energy balance.

| Variable | Description | Physical significance |
|---|---|---|
| `CO2_LBC` | Carbon dioxide (CO₂) | Absorbs outgoing infrared radiation and contributes to global warming |
| `CH4_LBC` | Methane (CH₄) | Powerful greenhouse gas |
| `N2O_LBC` | Nitrous oxide (N₂O) | Long-lived greenhouse gas |
| `CFC11eq_LBC` | CFC-11-equivalent concentration | Represents radiative effects of certain halogenated gases |
| `CF2CL2_LBC` | CFC-12 concentration | Long-lived greenhouse gas |

### Data structure

```python
ds["CO2_LBC"].dims
# ('forcing_scenario', 'time_year')

ds["CO2_LBC"].shape
# (1, 86)
```

These variables have **annual temporal resolution** and no latitude or longitude dimensions.

### Important Physics

A commonly used approximation for CO₂ radiative forcing is:

$$
\Delta F \approx 5.35 \ln\left(\frac{C}{C_0}\right)
$$

where:

- $C$: Current CO₂ concentration
- $C_0$: Reference CO₂ concentration
- $\Delta F$: Change in radiative forcing (W/m²)

## 2. Aerosol Forcings

Aerosols are small particles suspended in the atmosphere. They influence climate by scattering and absorbing sunlight and modifying cloud properties.

| Variable | Description |
|---|---|
| `BC_AX` | Black-carbon aerosol forcing |

The remaining aerosol variable names should be verified directly from the dataset.

### Data structure

```python
ds["BC_AX"].dims
# ('forcing_scenario', 'time_month', 'lat', 'lon')

ds["BC_AX"].shape
# (1, 1032, 12, 18)
```

Unlike the greenhouse-gas variables, aerosol forcings vary across **time and geographical location**.

## 3. Climate Target Variables

These are the seven variables that the ClimX emulator must predict.

| Variable | Physical meaning | Unit |
|---|---|---|
| `tas` | Near-surface air temperature | K |
| `tasmax` | Daily maximum near-surface temperature | K |
| `tasmin` | Daily minimum near-surface temperature | K |
| `pr` | Precipitation rate | kg m⁻² s⁻¹ |
| `huss` | Near-surface specific humidity | kg/kg |
| `psl` | Mean sea-level pressure | Pa |
| `sfcWind` | Near-surface wind speed | m/s |

### A. Temperature: `tas`, `tasmax`, `tasmin`

These represent different aspects of near-surface temperature:

- `tas`: Daily mean near-surface temperature
- `tasmax`: Daily maximum near-surface temperature
- `tasmin`: Daily minimum near-surface temperature

Temperature conversion:

$$
T_{^\circ C} = T_K - 273.15
$$

A useful physical consistency check is:

$$
T_{\min} \leq T_{\mathrm{mean}} \leq T_{\max}
$$

### B. Precipitation: `pr`

Precipitation is stored as a mass flux per unit area.

To convert into millimeters per day:

$$
P_{\mathrm{mm/day}}
=
P_{\mathrm{kg/m^2/s}} \times 86400
$$

This conversion assumes liquid-water-equivalent precipitation.

### C. Specific Humidity: `huss`

Specific humidity measures the ratio of water-vapor mass to the total mass of moist air.

$$
q = \frac{m_{\mathrm{water\ vapor}}}{m_{\mathrm{moist\ air}}}
$$

Specific humidity differs from relative humidity, which measures how close the air is to saturation.

### D. Sea-Level Pressure: `psl`

Atmospheric pressure adjusted to mean sea level.

$$
101325\ \mathrm{Pa} = 1013.25\ \mathrm{hPa}
$$

This adjustment allows atmospheric pressure to be compared across locations at different elevations.

### E. Wind Speed: `sfcWind`

Near-surface wind speed represents the magnitude of horizontal wind velocity.

$$
|\mathbf{v}| = \sqrt{u^2 + v^2}
$$

Here, $u$ and $v$ represent the horizontal wind components.

## 4. Temporal and Spatial Structure

The variables have different dimensions because their temporal and spatial resolutions differ.

| Variable type | Dimensions | Temporal resolution |
|---|---|---|
| Greenhouse-gas forcings | `(forcing_scenario, time_year)` | Annual |
| Aerosol forcings | `(forcing_scenario, time_month, lat, lon)` | Monthly |
| Climate targets | `(forcing_scenario, time, lat, lon)` | Daily |

For the SSP1-2.6 dataset:

- Annual time points: **86**
- Monthly time points: **1,032**
- Daily time points: **31,389**
- Latitude points: **12**
- Longitude points: **18**

The spatial grid contains:

$$
N_{\mathrm{grid}} = 12 \times 18 = 216
$$

grid cells.

## 5. Relationship Between Inputs and Outputs

The climate model responds to external forcings through several interacting physical processes:

1. Radiative transfer and Earth's energy balance
2. Atmospheric thermodynamics
3. Atmospheric circulation and fluid dynamics
4. The hydrological cycle
5. Ocean–atmosphere interactions

The ClimX emulation problem can be represented schematically as:

$$
\text{External forcings}
\longrightarrow
\text{Climate system response}
\longrightarrow
\text{Daily climate fields}
$$

In mathematical notation:

$$
\mathbf{Y}(t,\phi,\lambda)
\approx
\mathcal{M}_{\theta}
\left(\mathbf{F}_{\leq t},\phi,\lambda,t\right)
$$

where:

- $\mathbf{Y}$: Predicted climate variables
- $\mathbf{F}_{\leq t}$: External forcing history up to time $t$
- $\phi$: Latitude
- $\lambda$: Longitude
- $t$: Time
- $\mathcal{M}_{\theta}$: Machine-learning emulator parameterized by $\theta$

This is a conceptual forcing-based formulation. Other emulators may incorporate previous climate states as additional inputs.

**Important:** External forcings influence the climate system, but they do not uniquely determine the daily atmospheric state.

## 6. Inspect Variables Using Xarray

The following code retrieves information directly from the dataset.

```python
for name, da in ds.data_vars.items():
    print(f"Variable: {name}")
    print(f"Description: {da.attrs.get('long_name', 'N/A')}")
    print(f"Units: {da.attrs.get('units', 'N/A')}")
    print(f"Dimensions: {da.dims}")
    print(f"Shape: {da.shape}")
    print()
```

This helps identify:

- The physical meaning of each variable
- Its units
- Its dimensions
- Its shape
- Whether it varies annually, monthly, or daily

## 7. Key Takeaways

1. ClimX contains **10 forcing variables** and **7 target variables**.
2. Greenhouse-gas forcing variables are provided annually.
3. Spatial aerosol forcing variables are provided monthly.
4. Climate target variables are provided daily on a latitude–longitude grid.
5. The emulator attempts to reproduce climate-model outputs from prescribed forcings, potentially using previous climate states.
6. Understanding the physical relationships between variables is important for developing accurate and physically consistent emulators.

In [6]:
filename = (
    "NorESM2-MM_r1i1p1f1_with_forcing_"
    "subsampled_16_train_ssp126.nc"
)

ds = xr.open_dataset(DATADIR / filename)

print(ds)

<xarray.Dataset> Size: 199MB
Dimensions:           (forcing_scenario: 1, time_year: 86, time_month: 1032,
                       lat: 12, lon: 18, time: 31389)
Coordinates:
  * forcing_scenario  (forcing_scenario) <U6 24B 'ssp126'
  * lat               (lat) float64 96B -82.93 -67.85 -52.77 ... 67.85 82.93
  * lon               (lon) float64 144B 9.375 29.38 49.38 ... 309.4 329.4 349.4
  * time              (time) object 251kB 2015-01-01 12:00:00 ... 2100-12-30 ...
  * time_month        (time_month) object 8kB 2015-01-15 00:00:00 ... 2100-12...
  * time_year         (time_year) object 688B 2015-07-02 12:00:00 ... 2100-07...
Data variables: (12/17)
    CO2_LBC           (forcing_scenario, time_year) float32 344B ...
    N2O_LBC           (forcing_scenario, time_year) float32 344B ...
    CH4_LBC           (forcing_scenario, time_year) float32 344B ...
    CFC11eq_LBC       (forcing_scenario, time_year) float32 344B ...
    CF2CL2_LBC        (forcing_scenario, time_year) float32 344B ...

In [11]:
ds['lat']

<xarray.DataArray 'lat' (lat: 12)> Size: 96B
array([-82.931937, -67.853403, -52.774869, -37.696335, -22.617801,  -7.539267,
         7.539267,  22.617801,  37.696335,  52.774869,  67.853403,  82.931937])
Coordinates:
  * lat      (lat) float64 96B -82.93 -67.85 -52.77 -37.7 ... 52.77 67.85 82.93
Attributes:
    axis:           Y
    bounds:         lat_bnds
    long_name:      Latitude
    standard_name:  latitude
    units:          degrees_north

In [5]:
# Data variable
print(ds.data_vars)

Data variables:
    CO2_LBC      (forcing_scenario, time_year) float32 344B ...
    N2O_LBC      (forcing_scenario, time_year) float32 344B ...
    CH4_LBC      (forcing_scenario, time_year) float32 344B ...
    CFC11eq_LBC  (forcing_scenario, time_year) float32 344B ...
    CF2CL2_LBC   (forcing_scenario, time_year) float32 344B ...
    BC_AX        (forcing_scenario, time_month, lat, lon) float64 2MB ...
    BC_N         (forcing_scenario, time_month, lat, lon) float64 2MB ...
    SO2          (forcing_scenario, time_month, lat, lon) float64 2MB ...
    SO4_PR       (forcing_scenario, time_month, lat, lon) float64 2MB ...
    OM_NI        (forcing_scenario, time_month, lat, lon) float64 2MB ...
    huss         (forcing_scenario, time, lat, lon) float32 27MB ...
    pr           (forcing_scenario, time, lat, lon) float32 27MB ...
    psl          (forcing_scenario, time, lat, lon) float32 27MB ...
    sfcWind      (forcing_scenario, time, lat, lon) float32 27MB ...
    tas          (

In [6]:
# Inspect a particular variable
print(ds['tas'])

<xarray.DataArray 'tas' (forcing_scenario: 1, time: 31389, lat: 12, lon: 18)> Size: 27MB
[6780024 values with dtype=float32]
Coordinates:
  * forcing_scenario  (forcing_scenario) <U6 24B 'ssp126'
  * lat               (lat) float64 96B -82.93 -67.85 -52.77 ... 67.85 82.93
  * lon               (lon) float64 144B 9.375 29.38 49.38 ... 309.4 329.4 349.4
  * time              (time) object 251kB 2015-01-01 12:00:00 ... 2100-12-30 ...
Attributes:
    cell_measures:  area: areacella
    cell_methods:   area: time: mean
    comment:        near-surface (usually, 2 meter) air temperature
    history:        2019-12-24T16:08:46Z altered by CMOR: Treated scalar dime...
    long_name:      Near-Surface Air Temperature
    original_name:  TREFHT
    standard_name:  air_temperature
    units:          K


In [7]:
# Dimension and shape of all variables
for name, da in ds.data_vars.items():
    print(name, da.dims, da.shape)

CO2_LBC ('forcing_scenario', 'time_year') (1, 86)
N2O_LBC ('forcing_scenario', 'time_year') (1, 86)
CH4_LBC ('forcing_scenario', 'time_year') (1, 86)
CFC11eq_LBC ('forcing_scenario', 'time_year') (1, 86)
CF2CL2_LBC ('forcing_scenario', 'time_year') (1, 86)
BC_AX ('forcing_scenario', 'time_month', 'lat', 'lon') (1, 1032, 12, 18)
BC_N ('forcing_scenario', 'time_month', 'lat', 'lon') (1, 1032, 12, 18)
SO2 ('forcing_scenario', 'time_month', 'lat', 'lon') (1, 1032, 12, 18)
SO4_PR ('forcing_scenario', 'time_month', 'lat', 'lon') (1, 1032, 12, 18)
OM_NI ('forcing_scenario', 'time_month', 'lat', 'lon') (1, 1032, 12, 18)
huss ('forcing_scenario', 'time', 'lat', 'lon') (1, 31389, 12, 18)
pr ('forcing_scenario', 'time', 'lat', 'lon') (1, 31389, 12, 18)
psl ('forcing_scenario', 'time', 'lat', 'lon') (1, 31389, 12, 18)
sfcWind ('forcing_scenario', 'time', 'lat', 'lon') (1, 31389, 12, 18)
tas ('forcing_scenario', 'time', 'lat', 'lon') (1, 31389, 12, 18)
tasmax ('forcing_scenario', 'time', 'lat', 'lo

In [16]:
# Variable's physical meaning
ds['tas']

<xarray.DataArray 'tas' (forcing_scenario: 1, time: 31389, lat: 12, lon: 18)> Size: 27MB
[6780024 values with dtype=float32]
Coordinates:
  * forcing_scenario  (forcing_scenario) <U6 24B 'ssp126'
  * lat               (lat) float64 96B -82.93 -67.85 -52.77 ... 67.85 82.93
  * lon               (lon) float64 144B 9.375 29.38 49.38 ... 309.4 329.4 349.4
  * time              (time) object 251kB 2015-01-01 12:00:00 ... 2100-12-30 ...
Attributes:
    cell_measures:  area: areacella
    cell_methods:   area: time: mean
    comment:        near-surface (usually, 2 meter) air temperature
    history:        2019-12-24T16:08:46Z altered by CMOR: Treated scalar dime...
    long_name:      Near-Surface Air Temperature
    original_name:  TREFHT
    standard_name:  air_temperature
    units:          K

In [25]:
# First 5 annual CO2 forcing values
print(ds["CO2_LBC"].values[0, :5])

[0.00039995 0.00040312 0.00040575 0.00040859 0.00041142]


In [18]:
# Temperature on the first day at all grid points
print(ds["tas"].dims)
print(ds["tas"].shape)
print(ds["tas"].isel(forcing_scenario=0, time=0).values)

('forcing_scenario', 'time', 'lat', 'lon')
(1, 31389, 12, 18)
[[249.65309 245.67001 243.6049  243.85922 242.85591 243.78828 245.87074
  249.99084 259.4757  262.428   262.7197  258.54745 255.09    256.01138
  262.5001  263.09338 260.16522 254.34691]
 [267.39688 266.91095 262.20435 267.18948 260.41156 259.77936 261.17172
  265.13113 272.25287 274.50406 273.62436 273.2683  273.26688 272.57135
  270.5253  272.82907 272.45193 270.46707]
 [276.50375 278.48438 278.1407  277.7496  279.9643  279.43945 278.5314
  279.67896 281.69724 281.10355 281.7815  283.05096 281.7227  279.6325
  281.03488 281.52783 278.9057  275.99832]
 [288.8395  292.7524  291.02917 290.31775 289.7223  288.66205 288.4729
  291.25485 290.7057  289.54727 290.17566 291.7878  289.16315 287.69266
  290.8589  294.1436  289.91617 286.4397 ]
 [294.23367 301.67755 299.96478 298.45685 296.07324 297.4389  306.4527
  301.04282 298.30685 296.5575  297.76526 298.19427 296.07483 293.16733
  294.53915 300.88638 296.32166 293.8127 ]
 [298.4

In [26]:
for name, da in ds.data_vars.items():
    print(f"Variable: {name}")
    print(f"Description: {da.attrs.get('long_name', 'N/A')}")
    print(f"Units: {da.attrs.get('units', 'N/A')}")
    print(f"Dimensions: {da.dims}")
    print(f"Shape: {da.shape}")
    print()

Variable: CO2_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 86)

Variable: N2O_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 86)

Variable: CH4_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 86)

Variable: CFC11eq_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 86)

Variable: CF2CL2_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 86)

Variable: BC_AX
Description: Emissions of BC_AX from AGRTRADOMSOLWSTSHP in anthrosurf
Units: molecules cm-2 s-1
Dimensions: ('forcing_scenario', 'time_month', 'lat', 'lon')
Shape: (1, 1032, 12, 18)

Variable: BC_N
Description: Emissions of BC_N from AGRTRADOMSOLWSTSHP in anthrosurf
Units: molecules cm-2 s-1
Dimensions: ('forcing_scenario', 'time_month', 'lat', 'lon')
Shape: (1, 1032, 12, 18)

Variable: SO2
Description: Emissions of SO2 from AGRTRADO

### Explore forcing data (subsampled_16)

subsampled_16: Each dimention (lat, lon) is subsampled by a factor of 16, resulting in a coarser spatial resolution.

In [27]:
filename = (
    "NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_forcings_all.nc"
)

ds = xr.open_dataset(DATADIR / filename)

print(ds)

<xarray.Dataset> Size: 36MB
Dimensions:           (forcing_scenario: 4, time_year: 86, time_month: 1032,
                       lat: 12, lon: 18, time: 31389)
Coordinates:
  * forcing_scenario  (forcing_scenario) <U6 96B 'ssp126' 'ssp245' ... 'ssp585'
  * lat               (lat) float64 96B -82.93 -67.85 -52.77 ... 67.85 82.93
  * lon               (lon) float64 144B 9.375 29.38 49.38 ... 309.4 329.4 349.4
  * time_month        (time_month) object 8kB 2015-01-15 00:00:00 ... 2100-12...
  * time_year         (time_year) object 688B 2015-07-02 12:00:00 ... 2100-07...
  * time              (time) object 251kB 2015-01-01 12:00:00 ... 2100-12-30 ...
Data variables:
    CO2_LBC           (forcing_scenario, time_year) float32 1kB ...
    N2O_LBC           (forcing_scenario, time_year) float32 1kB ...
    CH4_LBC           (forcing_scenario, time_year) float32 1kB ...
    CFC11eq_LBC       (forcing_scenario, time_year) float32 1kB ...
    CF2CL2_LBC        (forcing_scenario, time_year) float32

In [ ]:
for name, da in ds.data_vars.items():
    print(f"Variable: {name}")
    print(f"Description: {da.attrs.get('long_name', 'N/A')}")
    print(f"Units: {da.attrs.get('units', 'N/A')}")
    print(f"Dimensions: {da.dims}")
    print(f"Shape: {da.shape}")
    print()

Variable: CO2_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (4, 86)

Variable: N2O_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (4, 86)

Variable: CH4_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (4, 86)

Variable: CFC11eq_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (4, 86)

Variable: CF2CL2_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (4, 86)

Variable: BC_AX
Description: Emissions of BC_AX from AGRTRADOMSOLWSTSHP in anthrosurf
Units: molecules cm-2 s-1
Dimensions: ('forcing_scenario', 'time_month', 'lat', 'lon')
Shape: (4, 1032, 12, 18)

Variable: BC_N
Description: Emissions of BC_N from AGRTRADOMSOLWSTSHP in anthrosurf
Units: molecules cm-2 s-1
Dimensions: ('forcing_scenario', 'time_month', 'lat', 'lon')
Shape: (4, 1032, 12, 18)

Variable: SO2
Description: Emissions of SO2 from AGRTRADO

In [29]:
for name, coord in ds.coords.items():
    print(f"Coordinate: {name}")
    print(f"Description: {coord.attrs.get('long_name', 'N/A')}")
    print(f"Units: {coord.attrs.get('units', 'N/A')}")
    print(f"Dimensions: {coord.dims}")
    print(f"Shape: {coord.shape}")
    print(f"Data type: {coord.dtype}")
    print()

Coordinate: forcing_scenario
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario',)
Shape: (4,)
Data type: <U6

Coordinate: lat
Description: Latitude
Units: degrees_north
Dimensions: ('lat',)
Shape: (12,)
Data type: float64

Coordinate: lon
Description: Longitude
Units: degrees_east
Dimensions: ('lon',)
Shape: (18,)
Data type: float64

Coordinate: time_month
Description: N/A
Units: N/A
Dimensions: ('time_month',)
Shape: (1032,)
Data type: object

Coordinate: time_year
Description: N/A
Units: N/A
Dimensions: ('time_year',)
Shape: (86,)
Data type: object

Coordinate: time
Description: time
Units: N/A
Dimensions: ('time',)
Shape: (31389,)
Data type: object



In [30]:
filename = (
    "NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_hist_train_1932_2014.nc"
)

ds = xr.open_dataset(DATADIR / filename)

print(ds)

<xarray.Dataset> Size: 192MB
Dimensions:           (forcing_scenario: 1, time_month: 1980, lat: 12, lon: 18,
                       time_year: 165, time: 30294)
Coordinates:
  * forcing_scenario  (forcing_scenario) <U10 40B 'historical'
  * lat               (lat) float64 96B -82.93 -67.85 -52.77 ... 67.85 82.93
  * lon               (lon) float64 144B 9.375 29.38 49.38 ... 309.4 329.4 349.4
  * time              (time) object 242kB 1932-01-01 12:00:00 ... 2014-12-30 ...
  * time_month        (time_month) object 16kB 1850-01-17 00:00:00 ... 2014-1...
  * time_year         (time_year) object 1kB 1850-07-02 12:00:00 ... 2014-07-...
Data variables: (12/17)
    BC_AX             (forcing_scenario, time_month, lat, lon) float32 2MB ...
    BC_N              (forcing_scenario, time_month, lat, lon) float32 2MB ...
    CF2CL2_LBC        (forcing_scenario, time_year) float32 660B ...
    CFC11eq_LBC       (forcing_scenario, time_year) float32 660B ...
    CH4_LBC           (forcing_scenario, t

In [31]:
for name, da in ds.data_vars.items():
    print(f"Variable: {name}")
    print(f"Description: {da.attrs.get('long_name', 'N/A')}")
    print(f"Units: {da.attrs.get('units', 'N/A')}")
    print(f"Dimensions: {da.dims}")
    print(f"Shape: {da.shape}")
    print()

Variable: BC_AX
Description: Emissions of BC_AX from AGRTRADOMSOLWSTSHP in anthrosurf
Units: molecules cm-2 s-1
Dimensions: ('forcing_scenario', 'time_month', 'lat', 'lon')
Shape: (1, 1980, 12, 18)

Variable: BC_N
Description: Emissions of BC_N from AGRTRADOMSOLWSTSHP in anthrosurf
Units: molecules cm-2 s-1
Dimensions: ('forcing_scenario', 'time_month', 'lat', 'lon')
Shape: (1, 1980, 12, 18)

Variable: CF2CL2_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 165)

Variable: CFC11eq_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 165)

Variable: CH4_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 165)

Variable: CO2_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 165)

Variable: N2O_LBC
Description: N/A
Units: N/A
Dimensions: ('forcing_scenario', 'time_year')
Shape: (1, 165)

Variable: OM_NI
Description: Emissions of OM_NI from

In [36]:
ds['time'].shape

(30294,)